## 🎯 Learning Objectives
* Understand the necessity of integrating external tools with Large Language Models (LLMs) for enhanced capabilities.
* Learn how to define and integrate various types of tools (search, calculator, custom APIs) using LangChain's `Tool` abstraction.
* Implement a LangChain agent capable of dynamically selecting and utilizing appropriate tools to answer complex queries.
* Analyze the performance implications and typical use cases of tool-augmented LLM agents.


## Integrating Tools: Search, Calculators, and APIs

Large Language Models (LLMs) are incredibly powerful for natural language understanding and generation. However, they possess inherent limitations:

1.  **Knowledge Cut-off**: Their training data is static, meaning they lack real-time information about current events, stock prices, or the latest research.
2.  **Factual Hallucination**: While adept at generating coherent text, LLMs can sometimes confidently produce factually incorrect information.
3.  **Complex Reasoning**: While improving, LLMs can struggle with precise mathematical calculations or multi-step logical deductions that require external verification.
4.  **Interaction with External Systems**: LLMs cannot inherently interact with databases, send emails, or trigger actions in other software.

To overcome these limitations, we equip LLMs with **tools**. Think of an LLM as a brilliant, highly articulate scholar who is confined to a library of old books. While they can synthesize vast amounts of information from those books, they can't browse the internet for the latest news, use a calculator for precise sums, or send an email to a colleague. Tools are like giving this scholar:

*   **A Research Assistant (Search Tool)**: To find up-to-the-minute information from the internet, academic databases, or internal knowledge bases.
*   **A Calculator (Calculator Tool)**: To perform exact mathematical operations, ensuring accuracy beyond the LLM's probabilistic reasoning.
*   **A Personal Assistant (API Tool)**: To interact with external systems – booking flights, checking weather, updating CRM records, or querying a custom database.

LangChain provides a robust framework for defining and integrating these tools. The core idea is to wrap any function or external service into a `Tool` object, which an LLM agent can then intelligently choose to use based on the user's query. This transforms a static LLM into a dynamic, agentic system capable of real-world interaction and problem-solving.


In [ ]:
# Ensure you have the necessary packages installed:
# pip install langchain langchain-openai langchain-community tavily-python

import os
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain import agents
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# --- Configuration --- 
# Set your API keys. In a real application, use a secure secrets management system.
# For demonstration, we'll use environment variables.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

# Placeholder for API keys if not set in environment
if "OPENAI_API_KEY" not in os.environ:
    print("Warning: OPENAI_API_KEY not found. Please set it as an environment variable.")
    # For local testing without setting env var, uncomment and replace:
    # os.environ["OPENAI_API_KEY"] = "sk-..."

if "TAVILY_API_KEY" not in os.environ:
    print("Warning: TAVILY_API_KEY not found. Please set it as an environment variable.")
    # For local testing without setting env var, uncomment and replace:
    # os.environ["TAVILY_API_KEY"] = "tvly-..."

# --- 1. Define Tools --- 

# 1.1. Search Tool (using Tavily for real-time web search)
# Tavily is a search API specifically designed for LLM agents.
search_tool = TavilySearchResults(max_results=3)

# 1.2. Calculator Tool (a simple Python function wrapped as a tool)
def calculate_expression(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result. 
    Input should be a string like '2 + 2 * 3'."""
    try:
        # Using eval() for demonstration. For production, consider safer alternatives
        # or a dedicated math library to prevent arbitrary code execution.
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error calculating expression: {e}"

calculator_tool = Tool.from_function(
    func=calculate_expression,
    name="Calculator",
    description="Useful for performing mathematical calculations. Input should be a mathematical expression string."
)

# 1.3. Custom API Tool (simulating a weather API)
def get_current_weather(location: str) -> str:
    """Fetches the current weather for a specified location. 
    Input should be a city name, e.g., 'London'."""
    # In a real scenario, this would make an HTTP request to a weather API.
    # For this example, we'll return mock data.
    mock_weather_data = {
        "London": "22°C, Sunny",
        "New York": "18°C, Cloudy with a chance of rain",
        "Tokyo": "25°C, Humid",
        "Sydney": "28°C, Clear skies"
    }
    weather = mock_weather_data.get(location, "Weather data not available for this location.")
    return f"Current weather in {location}: {weather}"

weather_tool = Tool.from_function(
    func=get_current_weather,
    name="WeatherAPI",
    description="Useful for getting the current weather for a specific city. Input should be a city name."
)

# --- 2. Initialize the LLM --- 
# We'll use a powerful chat model like GPT-4o for better agentic reasoning.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- 3. Create the Agent --- 

# LangChain's `create_react_agent` is a common way to build agents.
# It uses the ReAct (Reasoning and Acting) framework.

# Define the prompt for the agent.
# The `tools` and `agent_scratchpad` placeholders are crucial for ReAct.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant with access to powerful tools. Use them wisely to answer the user's questions."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Combine all tools into a list
all_tools = [search_tool, calculator_tool, weather_tool]

# Create the agent executor
agent_executor = agents.create_react_agent(llm, all_tools, prompt)

# --- 4. Run the Agent with different queries --- 

print("--- Query 1: Search for real-time information ---")
response1 = agent_executor.invoke({"input": "What is the capital of France and what is the current year's inflation rate in the Eurozone?", "chat_history": []})
print(response1["output"])
print("\n" + "="*50 + "\n")

print("--- Query 2: Perform a calculation ---")
response2 = agent_executor.invoke({"input": "What is 12345 * (67 + 89) - 100?", "chat_history": []})
print(response2["output"])
print("\n" + "="*50 + "\n")

print("--- Query 3: Use a custom API ---")
response3 = agent_executor.invoke({"input": "What's the weather like in Tokyo right now?", "chat_history": []})
print(response3["output"])
print("\n" + "="*50 + "\n")

print("--- Query 4: Combine multiple tools ---")
response4 = agent_executor.invoke({"input": "What is the current population of Japan, and if it were to decrease by 0.5% next year, what would be the approximate population?", "chat_history": []})
print(response4["output"])
print("\n" + "="*50 + "\n")


### Interpreting the Output and Performance Considerations

When you run the code, you'll observe the agent's thought process (if `verbose=True` was set, which is often useful for debugging). The LLM first *reasons* about the user's query, *decides* which tool to use, *executes* the tool, *observes* the tool's output, and then *reasons again* to formulate a final answer. This iterative `Thought -> Action -> Observation` loop is the essence of the ReAct framework.

*   **Query 1 (Search)**: The agent identifies the need for current information (inflation rate) and correctly invokes the `TavilySearchResults` tool. It then synthesizes this with its internal knowledge (capital of France).
*   **Query 2 (Calculator)**: The agent recognizes a complex mathematical operation and delegates it to the `Calculator` tool, ensuring an accurate result.
*   **Query 3 (Custom API)**: The agent understands the request for real-time, location-specific data and calls the `WeatherAPI` tool.
*   **Query 4 (Combined Tools)**: This query demonstrates the agent's ability to chain tools. It first uses the `TavilySearchResults` to find Japan's current population, then uses the `Calculator` to perform the percentage decrease calculation.

#### Performance Trade-offs and Considerations:

1.  **Latency**: Each tool call involves an external API request, which adds latency to the overall response time. For applications requiring real-time interaction, minimizing tool calls or optimizing tool response times is crucial.
2.  **Cost**: Many external APIs (like search, weather, or custom enterprise APIs) have associated costs per call. Frequent or inefficient tool usage can quickly accumulate expenses.
3.  **Reliability**: The agent's performance is dependent on the reliability and availability of the external tools. If an API is down or returns an error, the agent's ability to answer might be compromised.
4.  **Security**: When integrating custom APIs, especially those that modify data or access sensitive information, robust security measures are paramount. The LLM's output could potentially be used to craft malicious inputs for tools if not properly sanitized and validated.
5.  **Tool Selection Accuracy**: The LLM's ability to correctly choose the right tool and format its input is critical. Poorly described tools or ambiguous queries can lead to incorrect tool usage or failure.
6.  **Observability**: Monitoring tool usage, success rates, and errors is essential for maintaining and improving agent performance in production.

### Typical Use Cases:

*   **Dynamic Q&A Systems**: Answering questions that require up-to-date information (e.g., "What's the latest news on X?", "What are the current stock prices for Y?").
*   **Data Analysis and Reporting**: Fetching data from databases or data warehouses, performing calculations, and generating summaries.
*   **Workflow Automation**: Interacting with CRM, ERP, or project management systems to update records, create tasks, or send notifications.
*   **Personal Assistants**: Scheduling meetings, sending emails, managing calendars, or controlling smart home devices.
*   **E-commerce and Customer Service**: Checking order status, looking up product information, or processing returns by interacting with backend systems.


### Resources

*   **LangChain Tools Documentation**: [https://python.langchain.com/docs/modules/agents/tools/](https://python.langchain.com/docs/modules/agents/tools/)
*   **LangChain Agents Documentation**: [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
*   **Tavily Search API**: [https://tavily.com/](https://tavily.com/)
*   **LangChain `create_react_agent`**: [https://api.python.langchain.com/en/latest/agents/langchain.agents.react.agent.create_react_agent.html](https://api.python.langchain.com/en/latest/agents/langchain.agents.react.agent.create_react_agent.html)
*   **OpenAI GPT-4o Model Information**: [https://openai.com/index/hello-gpt-4o/](https://openai.com/index/hello-gpt-4o/)
